# 🎬 Reels — Free-GPU Talking Avatar (MuseTalk)

Generate a talking-avatar reel from a **script + a short real talking-head video**, 100% free on a Colab **T4 GPU**. Driving with REAL footage (not a still photo) is what makes the mouth sharp and the head move naturally.

**Runtime → Change runtime type → GPU (T4)** before running, then run the cells top-to-bottom.

Pipeline: `script → your cloned voice (XTTS) → MuseTalk mouth-swap on your video → 9:16 reel`.

> MuseTalk's mmcv/mmpose only ship wheels for Python 3.10 + torch 2.1, but Colab is Python 3.12.
> So cell 3 installs an **isolated Miniforge Python-3.10 env** just for MuseTalk — the notebook
> kernel is left untouched (no restart needed).

In [ ]:
# 1) Check the GPU (must show a Tesla T4 / similar)
!nvidia-smi -L
import sys; print('python', sys.version.split()[0])

In [ ]:
# 2) Get this repo (tts.py, reel_utils.py, avatars/, examples/) + MuseTalk
import os
if not os.path.isdir('/content/reels'):
    !git clone https://github.com/amsinghnavdeep/reels.git /content/reels
%cd /content/reels
!git pull -q
!pip -q install edge-tts pydub pyyaml
os.makedirs('engines', exist_ok=True)
if not os.path.isdir('engines/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git engines/MuseTalk

In [ ]:
%%bash
# 3) Isolated Python-3.10 env for MuseTalk (~8-10 min, one-time).
#    torch 2.1.0+cu118 has PREBUILT mmcv/mmdet/mmpose wheels -> no slow source build.
set -e
MF=/opt/conda
if [ ! -x $MF/bin/conda ]; then
  wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
  bash /tmp/mf.sh -b -p $MF
fi
source $MF/etc/profile.d/conda.sh
conda create -y -n muse python=3.10 >/dev/null 2>&1 || true
conda activate muse
cd /content/reels/engines/MuseTalk
# MuseTalk deps EXCEPT torch/mm* (we pin those to versions that have wheels)
grep -viE '^(torch|torchvision|torchaudio|mmcv|mmdet|mmpose|mmengine)' requirements.txt > /tmp/req.txt || cp requirements.txt /tmp/req.txt
pip -q install -r /tmp/req.txt
pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118
pip -q install -U openmim
mim install mmengine
mim install 'mmcv==2.1.0'
mim install 'mmdet==3.1.0'
# chumpy (legacy mmpose dep) fails to build under pip build isolation -> install it first, no isolation
pip -q install "setuptools<70" wheel
pip -q install --no-build-isolation chumpy
mim install 'mmpose==1.1.0'
# mmpose.apis imports mmdet, whose __init__ over-strictly caps mmcv < 2.1.0; relax it (2.1.0 works fine here)
sed -i "s/mmcv_maximum_version = '2.1.0'/mmcv_maximum_version = '2.2.0'/" /opt/conda/envs/muse/lib/python3.10/site-packages/mmdet/__init__.py
export MPLBACKEND=Agg
python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"
python -c "from mmpose.apis import inference_topdown; print('mmpose import OK')"

In [ ]:
# 4) Download MuseTalk weights (~5 GB, one-time).
#    MuseTalk's download_weights.sh uses `huggingface-cli`, which the new
#    huggingface_hub REMOVED (silently downloads nothing) -> use the Python API.
%cd /content/reels/engines/MuseTalk
import os
from huggingface_hub import hf_hub_download
os.makedirs('models/face-parse-bisent', exist_ok=True)
def dl(repo, filename, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    p = hf_hub_download(repo_id=repo, filename=filename, local_dir=local_dir)
    print('  ok:', p)
for f in ['musetalk/musetalk.json', 'musetalk/pytorch_model.bin',
          'musetalkV15/musetalk.json', 'musetalkV15/unet.pth']:
    dl('TMElyralab/MuseTalk', f, 'models')
for f in ['config.json', 'diffusion_pytorch_model.bin']:
    dl('stabilityai/sd-vae-ft-mse', f, 'models/sd-vae')
for f in ['config.json', 'pytorch_model.bin', 'preprocessor_config.json']:
    dl('openai/whisper-tiny', f, 'models/whisper')
dl('yzd-v/DWPose', 'dw-ll_ucoco_384.pth', 'models/dwpose')
# Two non-HF files (these already worked via gdown/curl):
!gdown 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O models/face-parse-bisent/79999_iter.pth
!curl -sL https://download.pytorch.org/models/resnet18-5c106cde.pth -o models/face-parse-bisent/resnet18-5c106cde.pth
import glob
print('\nweights present:')
for p in sorted(glob.glob('models/**/*', recursive=True)):
    if os.path.isfile(p): print(' ', p, round(os.path.getsize(p)/1e6, 1), 'MB')

In [ ]:
# 5) Upload your driving VIDEO (real talking-head clip) + your VOICE sample, then clone your voice.
#    Realism comes from REAL footage — MuseTalk only swaps the mouth to your new audio.
%cd /content/reels
import os, subprocess
from google.colab import files
os.makedirs('output', exist_ok=True)
# OUTPUT LENGTH = your SCRIPT length, NOT the clip length. A short clip (even 10-30s) is fine:
# MuseTalk loops the driving frames to cover the whole cloned-voice audio. Make the reel longer
# by writing a longer examples/script.txt — no need for a longer input video.
print('Upload your talking-head VIDEO (.mp4, ~10-30s is plenty) ...')
DRIVE_VIDEO = next(iter(files.upload()))
print('Upload your VOICE sample (.m4a/.wav/.mp3, 8-60s) ...')
VOICE_SRC = next(iter(files.upload()))
SCRIPT = 'examples/script.txt'   # edit examples/script.txt (double-click it in the file browser) with your words

# Optional: trim to a clean talking-head window and crop out any burned captions/logo.
#   For a 360x640 short with bottom captions, e.g.:  TRIM='-ss 15 -to 19.5'  CROP='crop=360:360:0:48'
TRIM = ''
CROP = ''
# ALWAYS normalise to a safe path: spaces / # / () in the upload name break MuseTalk's
# ffmpeg frame extraction -> 0 frames read -> 'Error ... division by zero'. TRIM/CROP applied here too.
# cap the long side at 1280px: MuseTalk loads EVERY frame into RAM, so a 4K clip (2160x3840)
# exhausts Colab's memory and the kernel gets killed mid-run (shows up as a '^C').
_dn = "scale=1280:1280:force_original_aspect_ratio=decrease:force_divisible_by=2"
vf = f'-vf "{CROP},{_dn}"' if CROP else f'-vf "{_dn}"'
subprocess.run(f'ffmpeg -y {TRIM} -i "{DRIVE_VIDEO}" {vf} -an -r 25 -c:v libx264 -pix_fmt yuv420p output/drive_clip.mp4', shell=True, check=True)
DRIVE_VIDEO = 'output/drive_clip.mp4'
print('driving video:', DRIVE_VIDEO)

# normalise the voice sample to 16k mono wav
subprocess.run(f'ffmpeg -y -i "{VOICE_SRC}" -ar 16000 -ac 1 /content/reels/output/voice_ref.wav', shell=True, check=True)

# clone your voice with XTTS-v2 in an ISOLATED py3.10 env
#   (Colab's Python 3.12 kernel + numpy 2 break coqui-tts; py3.10 is the supported combo)
!source /opt/conda/etc/profile.d/conda.sh && \
  (conda env list | grep -q '/envs/tts$' || conda create -y -q -n tts python=3.10) && \
  conda activate tts && \
  export MPLBACKEND=Agg && \
  pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 && \
  pip -q install transformers==4.35.2 && \
  pip -q install coqui-tts==0.22.1 && \
  COQUI_TOS_AGREED=1 python /content/reels/tts.py --script /content/reels/examples/script.txt \
    --engine xtts --clone /content/reels/output/voice_ref.wav --lang hi \
    --output /content/reels/output/speech_raw.wav
#   Prefer a clean non-cloned Hindi voice instead? (no isolated env needed)
#   !python tts.py --script "$SCRIPT" --engine edge --voice hi-IN-MadhurNeural --output output/speech_raw.wav
import os; assert os.path.exists('output/speech_raw.wav'), 'TTS failed \u2014 see the logs above'
# TIMING FIX: strip leading/trailing silence so the mouth starts moving exactly with the audio
subprocess.run('ffmpeg -y -i output/speech_raw.wav -af '
  '"silenceremove=start_periods=1:start_silence=0.05:start_threshold=-45dB:'
  'stop_periods=-1:stop_silence=0.2:stop_threshold=-45dB" '
  '-ar 16000 -ac 1 output/speech.wav', shell=True, check=True)
from IPython.display import Audio; Audio(filename='output/speech.wav')


In [ ]:
# 6) Run MuseTalk lip-sync in the isolated env (your video + cloned audio -> talking video)
import yaml, os
os.chdir('/content/reels/engines/MuseTalk')
os.environ['FFMPEG_PATH'] = '/usr/bin'
os.environ['MPLBACKEND'] = 'Agg'   # Colab sets matplotlib_inline backend, invalid in the muse env
cfg = {'task_0': {'video_path': '/content/reels/' + DRIVE_VIDEO, 'audio_path': '/content/reels/output/speech.wav'}}
os.makedirs('configs/inference', exist_ok=True)
open('configs/inference/reels.yaml', 'w').write(yaml.dump(cfg))
# MuseTalk sets save_dir_full only for video input, but cleans it up unconditionally
# -> crashes on a still-image avatar AFTER the mp4 is written. Guard the cleanup.
!sed -i 's|^            shutil.rmtree(save_dir_full)|            if get_file_type(video_path) == "video": shutil.rmtree(save_dir_full)|' scripts/inference.py
!MPLBACKEND=Agg /opt/conda/envs/muse/bin/python -m scripts.inference \
  --inference_config configs/inference/reels.yaml \
  --result_dir /content/reels/output/muse \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 --fps 25 --batch_size 4 --use_float16 \
  --parsing_mode jaw --extra_margin 10

In [ ]:
# 6b) SHARPEN — restore the soft MuseTalk mouth patch back to input-quality with GFPGAN
#     (face restore) + Real-ESRGAN (frame upscaler). Runs in the same py3.10 'muse' env.
import os
os.chdir('/content/reels/engines/MuseTalk') if os.path.isdir('/content/reels') else os.chdir('/kaggle/working/reels/engines/MuseTalk')
RAW = None
import glob
cands = sorted(glob.glob('/content/reels/output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
assert cands, 'No MuseTalk output found \u2014 run cell 6 first.'
RAW = cands[-1]; print('raw MuseTalk clip:', RAW)

# one-time: install GFPGAN + Real-ESRGAN into the muse env. basicsr imports
# torchvision.transforms.functional_tensor, removed in torchvision 0.16 -> patch to functional.
!source /opt/conda/etc/profile.d/conda.sh && conda activate muse && \
  pip -q install gfpgan realesrgan basicsr facexlib && \
  BS=$(/opt/conda/envs/muse/bin/python -c "import basicsr,os;print(os.path.dirname(basicsr.__file__))") && \
  sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' \
    $BS/data/degradations.py 2>/dev/null; echo patched

# split MuseTalk clip into frames, GFPGAN-restore each (with Real-ESRGAN background), reassemble + mux audio.
os.makedirs('/content/reels/output/enh_in', exist_ok=True)
os.makedirs('/content/reels/output/enh_out', exist_ok=True)
!rm -f /content/reels/output/enh_in/* /content/reels/output/enh_out/*
!ffmpeg -y -i "$RAW" -qscale:v 1 /content/reels/output/enh_in/f%06d.png
# GFPGAN v1.4 restores the face (mouth/teeth) at 2x; we fit back to 1080x1920 in cell 7.
# write the restore script to a file (heredocs are unreliable inside a '!' cell)
open('/content/reels/output/_restore.py', 'w').write(r'''
import os, glob, cv2
from gfpgan import GFPGANer
root = "/content/reels"
gfp = GFPGANer(model_path="https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
               upscale=2, arch="clean", channel_multiplier=2, bg_upsampler=None)
frames = sorted(glob.glob(root + "/output/enh_in/*.png"))
for i, fp in enumerate(frames):
    img = cv2.imread(fp, cv2.IMREAD_COLOR)
    _, _, out = gfp.enhance(img, has_aligned=False, only_center_face=True, paste_back=True)
    cv2.imwrite(root + "/output/enh_out/" + os.path.basename(fp), out)
    if i % 50 == 0: print("  restored", i, "/", len(frames))
print("done", len(frames), "frames")
''')
!source /opt/conda/etc/profile.d/conda.sh && conda activate muse && \
  MPLBACKEND=Agg python /content/reels/output/_restore.py
# reassemble restored frames at 25fps and mux the cloned audio
!ffmpeg -y -framerate 25 -i /content/reels/output/enh_out/f%06d.png -i /content/reels/output/speech.wav \
  -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest /content/reels/output/muse_hd.mp4
print('enhanced clip:', '/content/reels/output/muse_hd.mp4')


In [ ]:
# 7) Compose a 9:16 reel + preview/download (prefers the GFPGAN-restored clip)
%cd /content/reels
import glob, os
hd = 'output/muse_hd.mp4'
if os.path.exists(hd):
    raw = hd
else:
    clips = sorted(glob.glob('output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
    assert clips, 'No MuseTalk output found \u2014 check the previous cell logs.'
    raw = clips[-1]
print('composing from:', raw)
!python reel_utils.py --in "$raw" --out output/reel.mp4
from google.colab import files
from IPython.display import Video
files.download('output/reel.mp4')
Video('output/reel.mp4', embed=True, width=270)


In [ ]:
# 8) DASHBOARD ASSETS — upload the driving video + voice ONCE for the dashboard runner.
#    (No cloud storage: these stay on this box. Swap them anytime by re-running this cell,
#     or instead paste public URLs in the dashboard's Source section.)
import os, shutil
os.makedirs('/content/reels/reels_state', exist_ok=True)
try:
    from google.colab import files
    print('Upload driving VIDEO (.mp4) ...'); v = next(iter(files.upload()))
    print('Upload VOICE sample (.m4a/.wav/.mp3) ...'); a = next(iter(files.upload()))
    shutil.move(v, '/content/reels/reels_state/video' + os.path.splitext(v)[1])
    shutil.move(a, '/content/reels/reels_state/voice' + os.path.splitext(a)[1])
    print('saved assets for the dashboard runner')
except Exception as e:
    print('skip (not on Colab or cancelled):', e)


In [ ]:
# 8) DASHBOARD RUNNER (optional) — connect this GPU box to your Cloudflare dashboard.
#    Deploy the UI+API first (see worker/DEPLOY.md), then paste the two values below.
#    Leave this cell running: it polls the dashboard and renders whatever you queue there.
import os
os.environ['API'] = 'https://reels-api.<YOUR-SUBDOMAIN>.workers.dev'  # your Worker URL
os.environ['RUNNER_KEY'] = '<YOUR-RUNNER-KEY>'                        # matches the Worker secret
!cd /content/reels && pip -q install instagrapi >/dev/null 2>&1; python runner.py
